In [1]:
import sys, tensorflow as tf
from PIL import Image
import numpy as np
import keras
import os
from keras.applications import MobileNetV2

In [2]:
data_dir = "/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/model-creation/images"
test_image = "/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/model-creation/test.png"

print("Executable:", sys.executable)
print("TF Version:", tf.__version__)
print(tf.config.list_physical_devices('GPU'))

Executable: /Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/.venv/bin/python
TF Version: 2.18.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
converted_count = 0
deleted_count = 0

for dir in os.listdir(data_dir):
    if dir == ".DS_Store":
        continue
    for file in os.listdir(os.path.join(data_dir, dir)):
        if file == ".DS_Store":
            continue
        allowed_exts = ["jpeg", "png", "gif", "bmp"]
        try:
            with Image.open(os.path.join(data_dir, dir, file)) as image_loaded:
                image_loaded.verify()

            with Image.open(os.path.join(data_dir, dir, file)) as image_loaded:
                acutal_ext = image_loaded.format.lower()
                if acutal_ext not in allowed_exts:
                    image_loaded.save(
                        os.path.join(data_dir, dir, file.split(".")[0] + ".png"),
                        "PNG",
                        icc_profile=None,
                    )
                    os.remove(os.path.join(data_dir, dir, file))
                    converted_count += 1
                    print(f"Converted {file} to PNG")
                    file = os.path.join(data_dir, dir, file.split(".")[0] + ".png")
                    if image_loaded.mode in ("RGBA", "P", "CMYK"):
                        image_loaded = image_loaded.convert("RGB")
                else:
                    image_loaded.save(
                        os.path.join(data_dir, dir, file),
                        image_loaded.format,
                        icc_profile=None,
                    )
        except Exception as e:
            os.remove(os.path.join(data_dir, dir, file))
            deleted_count += 1
            print(f"Deleted {file} due to error")

print(f"Converted: {converted_count}")
print(f"Deleted: {deleted_count}")

Converted: 0
Deleted: 0


In [4]:
data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.2),
        keras.layers.RandomFlip("vertical"),
        keras.layers.RandomContrast(0.2),
        keras.layers.RandomTranslation(0.1, 0.1)
    ]
)

2026-11-15 20:42:57.055991: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-11-15 20:42:57.056020: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-11-15 20:42:57.056024: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
I0000 00:00:1794796977.056112 2210596 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1794796977.056150 2210596 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [5]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

class_names = train_ds.class_names
num_classes = len(class_names)

Found 2791 files belonging to 17 classes.
Using 2233 files for training.
Found 2791 files belonging to 17 classes.
Using 558 files for validation.


In [6]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)

base_model.trainable = False

In [7]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [8]:
model = keras.Sequential(
    [
        data_augmentation,
        keras.layers.Lambda( 
            lambda x: keras.applications.mobilenet_v2.preprocess_input(x)
        ),
        base_model,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

In [9]:
model.compile(
    loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
)

model.fit(train_ds, validation_data=val_ds, epochs=15)

Epoch 1/15


/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-11-15 20:42:59.172809: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


70/70 ━━━━━━━━━━━━━━━━━━━━ 20s 245ms/step - accuracy: 0.2911 - loss: 2.3972 - val_accuracy: 0.4928 - val_loss: 1.7347
Epoch 2/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 16s 227ms/step - accuracy: 0.4917 - loss: 1.6930 - val_accuracy: 0.5502 - val_loss: 1.5006
Epoch 3/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 16s 225ms/step - accuracy: 0.5714 - loss: 1.4360 - val_accuracy: 0.5878 - val_loss: 1.3628
Epoch 4/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 16s 226ms/step - accuracy: 0.6037 - loss: 1.2967 - val_accuracy: 0.6057 - val_loss: 1.3065
Epoch 5/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 16s 224ms/step - accuracy: 0.6350 - loss: 1.2127 - val_accuracy: 0.6290 - val_loss: 1.2637
Epoch 6/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 17s 242ms/step - accuracy: 0.6458 - loss: 1.1771 - val_accuracy: 0.6308 - val_loss: 1.2453
Epoch 7/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 16s 230ms/step - accuracy: 0.6588 - loss: 1.1081 - val_accuracy: 0.6254 - val_loss: 1.2246
Epoch 8/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 217ms/step - accuracy: 0.6735 - loss: 1.0405 - val_accuracy: 0.634

In [10]:
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

In [11]:
custom_omtimizer = keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=custom_omtimizer,
    metrics=["accuracy"]
)

In [13]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    initial_epoch=15,
    callbacks=[early_stopping]
)

model.save("waste_classification_model.keras")

Epoch 16/40


/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


70/70 ━━━━━━━━━━━━━━━━━━━━ 25s 359ms/step - accuracy: 0.9064 - loss: 0.3028 - val_accuracy: 0.6649 - val_loss: 1.4437
Epoch 17/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 27s 379ms/step - accuracy: 0.9185 - loss: 0.2699 - val_accuracy: 0.6541 - val_loss: 1.4947
Epoch 18/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 27s 375ms/step - accuracy: 0.9288 - loss: 0.2371 - val_accuracy: 0.6613 - val_loss: 1.4778
Epoch 19/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 26s 370ms/step - accuracy: 0.9391 - loss: 0.2278 - val_accuracy: 0.6398 - val_loss: 1.6338
Epoch 20/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 29s 404ms/step - accuracy: 0.9409 - loss: 0.2164 - val_accuracy: 0.6416 - val_loss: 1.6702
Epoch 21/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 32s 443ms/step - accuracy: 0.9485 - loss: 0.1874 - val_accuracy: 0.6470 - val_loss: 1.5321
